# HOSA data pipeline

Private CPU run. Enable Internet and execute each code cell in order in the Draft session. Downloads NASA PGDA data, requests JPL Horizons, runs scientific checks, and writes a compact ZIP. The epoch 2026-11-14T00:00Z and G0 tolerances are PROVISIONAL.

In [ ]:
from pathlib import Path, PurePosixPath
import base64, io, os, sys, zipfile
kaggle=Path("/kaggle/working").is_dir()
root=Path(os.environ.get("LUNARSIGHT_ROOT", "/kaggle/temp/lunarsight-data" if kaggle else str(Path.cwd()/"lunarsight-run"))).resolve()
root.mkdir(parents=True,exist_ok=True)
archive = '''...
AAAA'''
with zipfile.ZipFile(io.BytesIO(base64.b64decode(archive))) as src:
    for item in src.infolist():
        rel=PurePosixPath(item.filename)
        if rel.is_absolute() or ".." in rel.parts:
            raise ValueError("Unexpected notebook source path")
        target=root.joinpath(*rel.parts)
        target.parent.mkdir(parents=True,exist_ok=True)
        target.write_bytes(src.read(item))
sys.path.insert(0,str(root/"src"))
from build_notebooks import make
make(root)
print("HOSA working tree:", root)

In [ ]:
from pipeline import run
import shutil
status=run(root,offline=os.environ.get("LUNARSIGHT_OFFLINE")=="1")
out=Path("/kaggle/working") if kaggle else root.parent
bundle=root.parent/"lunarsight-data-deliverables.zip"
if bundle.resolve()!=(out/bundle.name).resolve():
    shutil.copy2(bundle,out/bundle.name)
for name in ("execution_status.json","pytest.log","scientific_validation.md","eda_summary.md","sensitivity.json"):
    src=root/"reports"/name
    if src.exists(): shutil.copy2(src,out/name)
print("BUNDLE:", out/bundle.name, (out/bundle.name).stat().st_size,"bytes")
print("PIPELINE_STATUS:",status["pipeline_status"])
assert status["pipeline_status"]=="PASS", "Partial run; inspect execution_status.json and ZIP for successful stages."

In [ ]:
from IPython.display import Image, display
for name in ("Site01_dem_eda.png","Site01_horizons.png","Site01_horizon.png","Site04_horizon.png"):
    p=root/"figures"/name
    if p.exists(): display(Image(filename=str(p),width=900))